In [3]:
import inspect
from typing import Callable, Any, Dict
from pydantic import create_model, BaseModel
import json

## Defining Tool Decorator
#### This decorator is responsible to fetch the metadata from the function.
- Name
- Description from Docstring
- Expected and required parameters from function signature.

In [ ]:

class FuncationMetadataTool:
    """Wraps a Python function with metadata for an LLM."""
    def __init__(self, func: Callable, name: str, description: str, args_schema: type[BaseModel]):
        self.func = func
        self.name = name
        self.description = description
        self.args_schema = args_schema

    def __call__(self, *args, **kwargs) -> Any:
        # Validates arguments against the Pydantic schema before execution
        validated_args = self.args_schema(**kwargs)
        return self.func(**validated_args.model_dump())

    def get_llm_schema(self) -> Dict[str, Any]:
        """Generates OpenAI-style tool definition schema."""
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": self.args_schema.model_json_schema()
            }
        }

def tool(func: Callable) -> FuncationMetadataTool:
    "Decorator to transform a function into a CustomTool."
    # 1. Extract name and description
    name = func.__name__
    description = func.__doc__ or "No description provided."
    
    # 2. Extract function signatures and type hints
    sig = inspect.signature(func)
    fields = {}
    
    for param_name, param in sig.parameters.items():
        if param_name == 'self':
            continue
        # Default to Any if no type hint is provided
        param_type = param.annotation if param.annotation != inspect.Parameter.empty else Any
        # Handle default values
        default_value = param.default if param.default != inspect.Parameter.empty else ...
        fields[param_name] = (param_type, default_value)
    
    # 3. Dynamically create a Pydantic model for input validation
    schema_name = f"{name} InputSchema"
    args_schema = create_model(schema_name, **fields)
    
    return FuncationMetadataTool(func, name, description, args_schema)


## Defining `get_weather_information` & `Calculator` functions as a Tools that can be passed to OpenAI styled LLM call

In [6]:
@tool
def get_weather_information(city: str):
    """
    Retrieve weather information for a supported city.
    Args:
        city (str): The name of the city.
    Returns:
        dict: A dictionary containing:
            - celsius (int): Temperature in degrees Celsius.
            - conditions (str): A brief description of the weather.
    """
    weather = {
        "tokyo": {"celsius": 22, "conditions": "partly cloudy"},
        "delhi": {"celsius": 34, "conditions": "clear skies"},
        "london": {"celsius": 15, "conditions": "light rain"},
    }
    return weather.get(city.lower())

@tool
def calculator(expression: str) -> str:
    """
    Calculates mathematical expressions using numexpr.
    
    Args:
        expression: A string mathematical expression (e.g., "5.6 * (5 + 10.5)").
        
    Returns:
        The result of the calculation as a string.
    """
    try:
        result = ne.evaluate(expression)
        return f"The result of '{expression}' is {result}"
    except Exception as e:
        return f"Error evaluating expression: {e}"


### Check get_weather_information schema metadata

In [7]:
print(json.dumps(get_weather_information.get_llm_schema(), indent=2))

{
  "type": "function",
  "function": {
    "name": "get_weather_information",
    "description": "\nRetrieve weather information for a supported city.\nArgs:\n    city (str): The name of the city.\nReturns:\n    dict: A dictionary containing:\n        - celsius (int): Temperature in degrees Celsius.\n        - conditions (str): A brief description of the weather.\n",
    "parameters": {
      "properties": {
        "city": {
          "title": "City",
          "type": "string"
        }
      },
      "required": [
        "city"
      ],
      "title": "get_weather_information InputSchema",
      "type": "object"
    }
  }
}


### Check calculator schema metadata

In [8]:
print(json.dumps(calculator.get_llm_schema(), indent=2))

{
  "type": "function",
  "function": {
    "name": "calculator",
    "description": "\nCalculates mathematical expressions using numexpr.\n\nArgs:\n    expression: A string mathematical expression (e.g., \"5.6 * (5 + 10.5)\").\n\nReturns:\n    The result of the calculation as a string.\n",
    "parameters": {
      "properties": {
        "expression": {
          "title": "Expression",
          "type": "string"
        }
      },
      "required": [
        "expression"
      ],
      "title": "calculator InputSchema",
      "type": "object"
    }
  }
}
